In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
import pandas as pd
from datetime import datetime, timedelta
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from statsmodels.distributions.empirical_distribution import ECDF
from scipy.stats import ks_2samp
from scipy import stats
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mtick
from lifelines import KaplanMeierFitter
from matching_case_control import *
import warnings
from preproces_prod3 import *
from scipy.stats import norm
import gspread
from oauth2client.service_account import ServiceAccountCredentials
warnings.filterwarnings("ignore")

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'
path_nirsecl = path_actual.parent.parent/'Nirse_cl' / 'Data'

with open(path_data/'lista_ruts_cardio.pkl', 'rb') as f:
    lista_ruts_cardio = pickle.load(f)

filtro_cardio = 'RUN.isin(@lista_ruts_cardio)'
filtro_prematuro = 'RUN.isin(@lista_ruts_preterms)'
filtro_any = '(RUN.isin(@lista_ruts_cardio)) | (RUN.isin(@lista_ruts_preterms))'

df_pf = pd.read_csv(path_data / "df_pf.csv")
df_historic = pd.read_csv(path_data / "df_historic.csv")
df_historic_cardio = pd.read_csv(path_data / "df_historic_cardio.csv")
df_historic_prematuro = pd.read_csv(path_data / "df_historic_prematuro.csv")
pf_nac = pd.read_csv(path_data / "pf_nac.csv")
pf_all = (pd.read_csv(path_data / "pf_all.csv").assign(
        FECHA_NAC = lambda x: pd.to_datetime(x.FECHA_NAC, format = 'mixed'),
        fecha_nac = lambda x: pd.to_datetime(x.FECHA_NAC, format = 'mixed'),
        year_nac = lambda x: x.FECHA_NAC.dt.year,
        month_nac = lambda x: x.FECHA_NAC.dt.month,
        elegibilidad_alt = lambda df: (df.year_nac + (df.month_nac >= 10).astype(int)),
        season = lambda x: np.where(x.month_nac.between(4, 9, inclusive='both'), 'in_season','pre_season'),
        ))
nac_filtered_vrs = pd.read_csv(path_data / "nac_filtered_vrs.csv")
nacimientos_per_year_cardio = pd.read_csv(path_data / "nacimientos_per_year_cardio.csv")
nacimientos_per_year_prematuro = pd.read_csv(path_data / "nacimientos_per_year_prematuro.csv")
nacimientos_per_year_any = pd.read_csv(path_data / "nacimientos_per_year_any.csv")

dict_nac_filt = {'cardio': nacimientos_per_year_cardio, 
                 'prematuro': nacimientos_per_year_prematuro, 
                 'any': nacimientos_per_year_any}

In [3]:
_, _, nac_filtered_vrs, _ = filtros_IH_nac(pf_all)

ruts perdidos por filtro semanas y peso:  4687
Droped intersex: 112
Datos perdidos por fecha ingreso menor a fecha nacimiento: 40224
vrs en los primeros 7 dias de nacer: 3
Ruts eliminados: 43026


In [3]:
pf_all_filtrado = pf_all.query('SEMANAS<32')
pf_all_on_hist = pf_all_filtrado.query('RUN.isin(@df_historic.RUN.unique())')[['RUN','elegibilidad_alt','season','FECHA_NAC']].rename(columns={'elegibilidad_alt':'elegibilidad'}).sort_values(by='RUN').drop_duplicates(subset='RUN').set_index('RUN',drop=True)
df_historic_on_nac = df_historic.query('RUN.isin(@pf_all_filtrado.RUN.unique())')[['RUN','elegibilidad','season','FECHA_NAC']].sort_values(by='RUN').drop_duplicates(subset='RUN').set_index('RUN',drop=True)
pf_all_on_hist.compare(df_historic_on_nac,)

elegibilidad          \
                                                           self   other   
RUN                                                                       
0009fdb7f7a08c5997864c27ad81676be3668a80ba39b31...          NaN     NaN   
0024fd1de12853db8c544a5fb947a95ee772ceb6d5d9020...       2018.0  2019.0   
003486c0332b3e4c48f878037cfaa1ec98e29f7f9b9b010...       2021.0  2022.0   
005d1184c752c7cb59d4ec747cc149bdc066f61164cefd0...          NaN     NaN   
00640b39a49719dd03606b038029a649bce681ea4876d3f...          NaN     NaN   
...                                                         ...     ...   
fe7ecbed68ea3c07f1403bb5e39040a00a07690518e903d...          NaN     NaN   
fe8a43c6ee58eb5a680c8fcea9520e182ca1bab6a1cf665...          NaN     NaN   
ff47ef895ff020e5c3eb1174fff3699caf736c35eb20f20...       2020.0  2019.0   
ff9c24f6463d676cba57dad6420c08bfef5c03340865780...          NaN     NaN   
fff1f08ae31e3f48af59ebdd745e108cc68914082c6b941...       2024.0  2023.0   

                                                        season              \
                                                          self       other   
RUN                                                                          
0009fdb7f7a08c5997864c27ad81676be3668a80ba39b31...         NaN         NaN   
0024fd1de12853db8c544a5fb947a95ee772ceb6d5d9020...   in_season  pre_season   
003486c0332b3e4c48f878037cfaa1ec98e29f7f9b9b010...         NaN         NaN   
005d1184c752c7cb59d4ec747cc149bdc066f61164cefd0...         NaN         NaN   
00640b39a49719dd03606b038029a649bce681ea4876d3f...   in_season  pre_season   
...                                                        ...         ...   
fe7ecbed68ea3c07f1403bb5e39040a00a07690518e903d...  pre_season   in_season   
fe8a43c6ee58eb5a680c8fcea9520e182ca1bab6a1cf665...   in_season  pre_season   
ff47ef895ff020e5c3eb1174fff3699caf736c35eb20f20...  pre_season   in_season   
ff9c24f6463d676cba57dad6420c08bfef5c03340865780...         NaN         NaN   
fff1f08ae31e3f48af59ebdd745e108cc68914082c6b941...         NaN         NaN   

                                                    FECHA_NAC              
                                                         self       other  
RUN                                                                        
0009fdb7f7a08c5997864c27ad81676be3668a80ba39b31... 2019-07-04  2019-04-07  
0024fd1de12853db8c544a5fb947a95ee772ceb6d5d9020... 2018-05-11  2018-11-05  
003486c0332b3e4c48f878037cfaa1ec98e29f7f9b9b010... 2021-03-11  2021-11-03  
005d1184c752c7cb59d4ec747cc149bdc066f61164cefd0... 2021-01-03  2021-03-01  
00640b39a49719dd03606b038029a649bce681ea4876d3f... 2023-05-03  2023-03-05  
...                                                       ...         ...  
fe7ecbed68ea3c07f1403bb5e39040a00a07690518e903d... 2023-01-06  2023-06-01  
fe8a43c6ee58eb5a680c8fcea9520e182ca1bab6a1cf665... 2023-09-03  2023-03-09  
ff47ef895ff020e5c3eb1174fff3699caf736c35eb20f20... 2019-11-04  2019-04-11  
ff9c24f6463d676cba57dad6420c08bfef5c03340865780... 2022-06-04  2022-04-06  
fff1f08ae31e3f48af59ebdd745e108cc68914082c6b941... 2023-12-02  2023-02-12  

[389 rows x 6 columns]

In [195]:
display(count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=criterio_pali_1))

,elegibilidad_alt,nacs_filtred,nacs,lost_filter,lost_filter_perc
0,2018,1811,2642,831,0.315
1,2019,3049,4228,1179,0.279
2,2020,2924,3925,1001,0.255
3,2021,2839,3799,960,0.253
4,2022,3012,4161,1149,0.276
5,2023,3049,4220,1171,0.277
6,2024,2790,3812,1022,0.268


In [4]:
df_nac_per_year_filt_pali

,year,nacimientos
0,2018,1811
1,2019,3049
2,2020,2924
3,2021,2839
4,2022,3012
5,2023,3049
6,2024,2792


In [4]:
#################################################################################################################### SECOND SIN PALI + THIRD ########################################################################
criterio_pali_1 = (
    pf_all
    .assign(
        criterio_pali_normal = lambda x: ((x.RUN.isin(lista_ruts_cardio)) | (x.SEMANAS<32) | (x.PESO<1500)).astype(int),
        pali = lambda x: np.where(x.criterio_pali_normal==1, 1, 
                                  np.where((x.year_nac==2023) & (x.month_nac.between(7, 9, inclusive='both')) & (x.SEMANAS<=34) & ((x.PESO<2500)), 1, 0))
            )
    #.query('pali==1') # ((SEMANAS<32) | (PESO<1500) | (RUN.isin(@lista_ruts_cardio))) | ((32<=SEMANAS<=34) & (PESO>2500)) | ((PESO>=1500) & (SEMANAS==35)) & ~(RUN.isin(@lista_ruts_cardio))
    .query('pali==0')
    .RUN
)

# '((32<=SEMANAS<=34) & (PESO>2500)) | ((PESO>=1500) & (SEMANAS==35)) & ~(RUN.isin(@lista_ruts_cardio))'
#

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=criterio_pali_1)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@criterio_pali_1)')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) | 
                                                         ((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad) & (df.season == "pre_season")) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table_display = round(
    df_historic
    .query('RUN.isin(@criterio_pali_1)')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) | ((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad) & (df.season == "pre_season")) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@criterio_pali_1)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .query('year==2022 | year==2023 | year==2024')
    .set_index('year')
    .T
,2)

VRS_table = round(
    VRS_table_display.T
    .reset_index()
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24 , #* x.nacimientos / nac_24# AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 ) * x.nacimientos / nac_24, #* x.nacimientos / nac_24 # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'third+ RSV season', 'Ratio (first:third+)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="1_pali_vs_3+2_sin_pali")
display(VRS_table_display)
display(Tabla_3_impacto)

elegibilidad_inyear  year  discart  first_season  second_season
0                    2024     1198          1163           1683


year,2022,2023,2024
first_season,5161.0,6779.0,1163.0
second_season,1062.0,1870.0,1683.0
nacimientos,176718.0,159867.0,149172.0
egresos_total,8255.0,12118.0,4977.0


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,2920.47,4240.4,
2,third+ RSV season,600.96,1169.72,
3,Ratio (first:third+),4.86,3.63,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,5483.19,4095.47,4789.33 (981.27)
6,Expected number of cases,9689.78,6547.3,8118.54 (2222.07)
7,Averted number of cases,8312.02,5300.92,6806.47 (2129.17)
8,Relative reduction (%),85.78,80.96,83.37 (3.41)
9,Averted number of cases per 1000,47.04,33.16,40.1 (9.81)


In [12]:
VRS_table_display

year,2022,2023,2024
first_season,107.0,135.0,44.0
second_season,48.0,43.0,51.0
nacimientos,1084.0,1290.0,1108.0
egresos_total,241.0,260.0,176.0


In [ ]:
VRS_table = round(
    df_historic
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) | ((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad) & (df.season == "pre_season")) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@criterio_pali_1)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    # .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
    #         second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
    #         ratio = lambda x: x.first_season_per / x.second_season_per )
    #.drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023 | year==2024')
    .set_index('year')
    .T
,2)
VRS_table

year,2022,2023,2024
first_season,209.0,266.0,98.0
second_season,107.0,133.0,102.0
nacimientos,3012.0,3049.0,2790.0
egresos_total,506.0,604.0,364.0


In [171]:
#################################################################################################################### SECOND CON PALI ########################################################################

criterio_pali_1 = (
    pf_all
    .query('(SEMANAS<32) | (PESO<1500) | (RUN.isin(@lista_ruts_cardio))' )
    .RUN
)

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=criterio_pali_1)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad) & (df.season == "in_season")) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad) & (df.season == "in_season")) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@criterio_pali_1)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Second* RSV season', 'Ratio (first:second*)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="1_pali_vs_2_con_pali")
Tabla_3_impacto
    

elegibilidad_inyear  year  discart  first_season  second_season
0                    2024      102            98             63


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,6938.91,8724.17,
2,Second* RSV season,2456.84,2623.81,
3,Ratio (first:second*),2.82,3.32,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,6367.74,7496.77,6932.26 (798.34)
6,Expected number of cases,191.8,228.58,210.19 (26.01)
7,Averted number of cases,86.0,121.48,103.74 (25.09)
8,Relative reduction (%),44.84,53.15,49.0 (5.88)
9,Averted number of cases per 1000,28.55,39.84,34.2 (7.98)


In [172]:
#################################################################################################################### SECOND All ########################################################################

criterio_pali_1 = (
    pf_all
    .query('(SEMANAS<32) | (PESO<1500) | (RUN.isin(@lista_ruts_cardio))' )
    .RUN
)

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=criterio_pali_1)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@criterio_pali_1)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Second RSV season', 'Ratio (first:second)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="1_pali_vs_2_all")
Tabla_3_impacto
    

elegibilidad_inyear  year  discart  first_season  second_season
0                    2024       61            98            104


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,6938.91,8724.17,
2,Second RSV season,3652.06,4558.87,
3,Ratio (first:second),1.9,1.91,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,7082.44,7119.71,7101.08 (26.35)
6,Expected number of cases,213.32,217.08,215.2 (2.66)
7,Averted number of cases,107.53,109.98,108.76 (1.73)
8,Relative reduction (%),50.4,50.66,50.53 (0.18)
9,Averted number of cases per 1000,35.7,36.07,35.89 (0.26)


In [173]:
#################################################################################################################### Third All ########################################################################

criterio_pali_1 = (
    pf_all
    .query('(SEMANAS<32) | (PESO<1500) | (RUN.isin(@lista_ruts_cardio))' )
    .RUN
)

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=criterio_pali_1)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic
    .query('RUN.isin(@criterio_pali_1)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@criterio_pali_1)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Third RSV season', 'Ratio (first:third)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="1_pali_vs_3_all")
Tabla_3_impacto


elegibilidad_inyear  year  discart  first_season  second_season
0                    2024      104            98             61


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,6938.91,8724.17,
2,Third RSV season,2357.24,2427.03,
3,Ratio (first:third),2.94,3.59,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,6427.96,7849.1,7138.53 (1004.9)
6,Expected number of cases,193.61,239.32,216.46 (32.32)
7,Averted number of cases,87.81,132.22,110.02 (31.4)
8,Relative reduction (%),45.36,55.25,50.3 (6.99)
9,Averted number of cases per 1000,29.15,43.37,36.26 (10.06)


In [174]:
########################################################################## CARDIO 1 vs 2 #########################################################
filtro = '(RUN.isin(@lista_ruts_cardio))'

egresos_2024 = (
    df_historic_cardio
    .query('year==2024')
    .query('2023<=elegibilidad<=2024')                
    #.assign(elegibilidad_inyear = lambda x: np.where(x.elegibilidad == x.year, 'first_season', 'second_season'))
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & df.ola_enfermedad,'first_season',
                                                np.where((df.elegibilidad + 1 == df.year) & df.ola_enfermedad,'second_season',
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic_cardio
    #.query('year!=2024')
    #.assign(elegibilidad_inyear = lambda x: np.where(x.elegibilidad == x.year, 'first_season', np.where(x.elegibilidad + 1 == x.year , 'second_season', 'discart')))
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & df.ola_enfermedad,'first_season',
                                                np.where((df.elegibilidad + 1 == df.year) & df.ola_enfermedad,'second_season',
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(dict_nac_filt['cardio'], on='year', how='left')
    .merge(df_historic_cardio.groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year== 2023')
    .set_index('year')
    .T
,2)

nac_24 = dict_nac_filt['cardio'].query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Second RSV season', 'Ratio (first:second)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="cardio_1_vs_2")
Tabla_3_impacto

year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,9963.1,10542.64,
2,Second RSV season,4797.05,4186.05,
3,Ratio (first:second),2.08,2.52,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,11451.26,13873.65,12662.46 (1712.89)
6,Expected number of cases,124.13,178.97,151.55 (38.78)
7,Averted number of cases,77.17,123.09,100.13 (32.47)
8,Relative reduction (%),62.17,68.77,65.47 (4.67)
9,Averted number of cases per 1000,71.19,95.42,83.3 (17.13)


In [175]:
########################################################################## CARDIO 1 vs 3 #########################################################
filtro = '(RUN.isin(@lista_ruts_cardio))'

egresos_2024 = (
    df_historic_cardio
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')                
    #.assign(elegibilidad_inyear = lambda x: np.where(x.elegibilidad == x.year, 'first_season', 'second_season'))
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & df.ola_enfermedad,'first_season',
                                                np.where((df.elegibilidad + 2 == df.year) & df.ola_enfermedad,'second_season',
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic_cardio
    #.query('year!=2024')
    #.assign(elegibilidad_inyear = lambda x: np.where(x.elegibilidad == x.year, 'first_season', np.where(x.elegibilidad + 1 == x.year , 'second_season', 'discart')))
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & df.ola_enfermedad,'first_season',
                                                np.where((df.elegibilidad + 2 == df.year) & df.ola_enfermedad,'second_season',
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(dict_nac_filt['cardio'], on='year', how='left')
    .merge(df_historic_cardio.groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = dict_nac_filt['cardio'].query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Third RSV season', 'Ratio (first:third)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="cardio_1_vs_3")
Tabla_3_impacto

year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,9963.1,10542.64,
2,Third RSV season,3136.53,1937.98,
3,Ratio (first:third),3.18,5.44,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,8897.11,15220.22,12058.66 (4471.11)
6,Expected number of cases,96.44,196.34,146.39 (70.64)
7,Averted number of cases,49.48,140.46,94.97 (64.33)
8,Relative reduction (%),51.31,71.54,61.42 (14.3)
9,Averted number of cases per 1000,45.65,108.88,77.26 (44.71)


In [179]:
########################################################################## prematuros_no_pali #########################################################

preterm_not_pali = (
    pf_all
    .query('((34<SEMANAS<37) | ((32<=SEMANAS<=34) & (PESO>2500)))' )
    .RUN
)

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=preterm_not_pali)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@preterm_not_pali)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic
    .query('RUN.isin(@preterm_not_pali)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@preterm_not_pali)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Second RSV season', 'Ratio (first:second)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="preterm_not_palis")
Tabla_3_impacto

elegibilidad_inyear  year  discart  first_season  second_season
0                    2024       89           124            188


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,4255.12,6087.95,
2,Second RSV season,1198.76,2507.36,
3,Ratio (first:second),3.55,2.43,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,6657.36,4557.01,5607.18 (1485.17)
6,Expected number of cases,727.52,479.81,603.66 (175.16)
7,Averted number of cases,592.35,349.57,470.96 (171.67)
8,Relative reduction (%),81.42,72.86,77.14 (6.05)
9,Averted number of cases per 1000,54.2,33.2,43.7 (14.85)


In [177]:
##################################################################################### preterms todos #####################################################################

preterm = (
    pf_all
    .query('(SEMANAS<37)' )
    .RUN
)

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=preterm)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@preterm)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic
    .query('RUN.isin(@preterm)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@preterm)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Second RSV season', 'Ratio (first:second)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="preterm")
Tabla_3_impacto

elegibilidad_inyear  year  discart  first_season  second_season
0                    2024      154           230            312


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,4728.83,6418.66,
2,Second RSV season,1606.47,2994.96,
3,Ratio (first:second),2.94,2.14,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,6062.66,4412.95,5237.8 (1166.52)
6,Expected number of cases,1003.85,699.89,851.87 (214.93)
7,Averted number of cases,752.15,458.8,605.48 (207.43)
8,Relative reduction (%),74.93,65.55,70.24 (6.63)
9,Averted number of cases per 1000,45.42,28.93,37.17 (11.66)


In [178]:
##################################################################################### preterms todos vs 3rd season #####################################################################

preterm = (
    pf_all
    .query('(SEMANAS<37)' )
    .RUN
)

df_nac_per_year_filt_pali = (
    count_nacs_filter(pf_all, nac_filtered_vrs, '(RUN.isin(@pali_ruts))',pali_ruts=preterm)[['elegibilidad_alt','nacs_filtred']]
    .rename(columns={'nacs_filtred':'nacimientos','elegibilidad_alt':'year'})
)

egresos_2024 = (
    df_historic
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@preterm)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                        'discart'))
    )
    .query('ola_enfermedad')
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
)

print(egresos_2024)

egresos_second_season_2024 = egresos_2024.second_season.values[0]

vrs_eleg_2024 = egresos_2024.first_season.values[0] #df_f_vrs.query('fechaIng_vrs<="2024-09-30"').query(filtro).event_vrs.sum() 

VRS_table = round(
    df_historic
    .query('RUN.isin(@preterm)')
   # .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) ,'second_season', #& (df.season == "in_season")
                                                         'discart'))   
    )
    .groupby(['elegibilidad_inyear','year'])
    .VRS1.size()
    .unstack(0)
    .reset_index()
    .drop(columns='discart')
    .merge(df_nac_per_year_filt_pali, on='year', how='left') ###############################
    .merge(df_historic.query('RUN.isin(@preterm)').groupby('year', as_index=False).size().rename(columns={'size':'egresos_total'}),on='year',how='left')
    .assign(first_season_per = lambda x: x.first_season * 100000/ x.nacimientos,
            second_season_per = lambda x: x.second_season * 100000/ x.nacimientos,
            ratio = lambda x: x.first_season_per / x.second_season_per )
    .drop(columns=['egresos_total','first_season','second_season'])
    .query('year==2022 | year==2023')
    .set_index('year')
    .T
,2)

nac_24 = df_nac_per_year_filt_pali.query('year==2024').nacimientos.values[0]

tabale_3_proxy = round(
    VRS_table.T.reset_index()
    .assign(vrs_noeleg_2024_per = lambda x: egresos_second_season_2024* 100000/ nac_24, #x.nacimientos, 
            exp_rate_per = lambda x: x.vrs_noeleg_2024_per * x.ratio, 
            exp_numer_cases = lambda x: (egresos_second_season_2024 * x.ratio) * x.nacimientos / nac_24, # AQUÍ ANTES NO NORMALIZABA POR NACS #x.exp_rate_per * nacidos_2024 / 100000,
            averted_num_cases = lambda x: x.exp_numer_cases - (vrs_eleg_2024 * x.nacimientos / nac_24), # AQUÍ ANTES NO NORMALIZABA POR NACS
            relative_reduction = lambda x: 100*x.averted_num_cases/x.exp_numer_cases, # NO INFLUYE LA NORMALIZACIÓN POR NACIMIENTOS (SE CANCELAN)
            averted_num_cases_per = lambda x: x.averted_num_cases * 1000 /x.nacimientos, #/ nac_24,#
            number_needed_to_immunise = lambda x: (1000/x.averted_num_cases_per).astype(int) #np.where(x.averted_num_cases_per>0, (1000/x.averted_num_cases_per).astype(int), 'b')
    )
    .drop(columns=['vrs_noeleg_2024_per'])
    
, 2)   

main_analisis = pd.DataFrame([tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio']).mean()])

df_tomain = tabale_3_proxy.drop(columns=['first_season_per','second_season_per','ratio'])

q1 = df_tomain.quantile(0.25).round(2)
q3 = df_tomain.quantile(0.75).round(2)
sd = df_tomain.std().round(2)
med = df_tomain.mean().round(2)

# formatted = med.astype(str) + " (" + round(q1,2).astype(str) + "-" + round(q3,2).astype(str) + ")"

formatted = med.astype(str) + " (" + round(sd,2).astype(str) + ")"

main_analisis = pd.DataFrame([formatted])
to_loc = main_analisis.year.values[0]
main_analisis.loc[main_analisis.year==to_loc, 'year'] = 'Main Analysis (sd)'

Tabla_3_impacto = (
    pd.concat([tabale_3_proxy, main_analisis], ignore_index=True)
    .set_index('year')
    .T
    .reset_index()
    .replace([np.inf, -np.inf], np.nan)
    .fillna('')
    .query('index!="nacimientos"')
)


Names_col = ['First RSV season', 'Third RSV season', 'Ratio (first:third)', 'Expected rate per 100000',
             'Expected number of cases','Averted number of cases','Relative reduction (%)',
             'Averted number of cases per 1000','Number needed to immunise']

Tabla_3_impacto = Tabla_3_impacto.assign(index = Names_col)[['index',2022,2023, 'Main Analysis (sd)']]

na_row = pd.DataFrame([[np.nan] * len(Tabla_3_impacto.columns)], columns=Tabla_3_impacto.columns)

# Divide el DataFrame original en dos partes e inserta la fila
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:3], na_row, Tabla_3_impacto.iloc[3:]], ignore_index=True)
Tabla_3_impacto.loc[3, 'index'] = 'Estimates for the 2024 RSV season'
Tabla_3_impacto = pd.concat([Tabla_3_impacto.iloc[:0], na_row, Tabla_3_impacto.iloc[0:]], ignore_index=True)
Tabla_3_impacto.loc[0, 'index'] = 'RSV hospitalisation rate per 100000'

Tabla_3_impacto = Tabla_3_impacto.replace([np.inf, -np.inf], np.nan).fillna('')

# with pd.ExcelWriter(path_data/"tablas_impact_risk.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     Tabla_3_impacto.to_excel(writer, sheet_name="preterm_vs_3rd")
Tabla_3_impacto

elegibilidad_inyear  year  discart  first_season  second_season
0                    2024      312           230            154


year,index,2022,2023,Main Analysis (sd)
0,RSV hospitalisation rate per 100000,,,
1,First RSV season,4728.83,6418.66,
2,Third RSV season,821.36,1122.32,
3,Ratio (first:third),5.76,5.72,
4,Estimates for the 2024 RSV season,,,
5,Expected rate per 100000,5862.79,5822.08,5842.44 (28.79)
6,Expected number of cases,970.76,923.38,947.07 (33.5)
7,Averted number of cases,719.05,682.28,700.66 (26.0)
8,Relative reduction (%),74.07,73.89,73.98 (0.13)
9,Averted number of cases per 1000,43.43,43.02,43.22 (0.29)


In [4]:
df_vrs_match_case, _ = call_data('NAC_RNI_EGRESOS_ENTREGA_ISCI_11_04_2025_encr.csv')

n_rows_inicial= 157557
['DESCONOCIDO---->None' 'Región De Antofagasta---->ANTOFAGASTA'
 'Región De Arica Parinacota---->ARICA Y PARINACOTA'
 'Región De Atacama---->ATACAMA'
 'Región De Aysén del General Carlos Ibañez del Campo---->AISEN'
 'Región De Coquimbo---->COQUIMBO' 'Región De La Araucanía---->ARAUCANIA'
 'Región De Los Lagos---->LOS LAGOS' 'Región De Los Ríos---->LOS RIOS'
 'Región De Magallanes y de la Antártica Chilena---->MAGALLANES Y ANTARTICA'
 'Región De Tarapacá---->TARAPACA' 'Región De Valparaíso---->VALPARAISO'
 'Región De Ñuble---->NUBLE' 'Región Del Bíobío---->BIOBIO'
 "Región Del Libertador Gral. B. O'Higgins---->O'HIGGINS"
 'Región Del Maule---->MAULE'
 'Región Metropolitana de Santiago---->METROPOLITANA']
n_rows_post_prefiltred= 157557
Datos perdidos por muertes:  1227
ruts perdidos por filtro semanas y peso:  491
Droped intersex: 14
Datos perdidos por edad madre atípica: 2
Datos perdidos por fecha ingreso menor a fecha nacimiento: 19
vrs en los primeros 7 dias de 

In [6]:
query_df = df_vrs_match_case.query('(SEMANAS<=35) & ~(RUN.isin(@lista_ruts_cardio) | (SEMANAS<32) | (PESO<1500))')